# Exploratory Data Analysis

Superstore end-to-end analytics project notebook.

## Load cleaned data

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv('../Dataset/Cleaned_Data.csv', parse_dates=['order_date','ship_date','order_month'])
print(df.shape)
df.head()


## KPI snapshot
Revenue, profit, orders, customers, AOV, and profit margin summarize business performance.

In [ ]:

summary = {
    'Revenue': df['revenue'].sum(),
    'Profit': df['profit'].sum(),
    'Orders': df['order_id'].nunique(),
    'Customers': df['customer_id'].nunique(),
    'AOV': df.groupby('order_id')['revenue'].sum().mean(),
    'Margin %': df['profit'].sum()/df['revenue'].sum()
}
pd.Series(summary).map(lambda x: f'{x:,.2f}')


## Descriptive statistics
Sales and profit are right-skewed, typical for retail order-line data.

In [ ]:

# Descriptive statistics
numeric_profile = df[['sales','quantity','discount','profit','ship_days','profit_margin']].describe().T
numeric_profile


## Correlation analysis
Discount generally has a negative relationship with profit and margin.

In [ ]:

# Correlation analysis
corr = df[['sales','quantity','discount','profit','ship_days','profit_margin']].corr(numeric_only=True)
corr


## Trend analysis
Monthly trend helps identify seasonality and growth patterns.

In [ ]:

monthly = df.groupby('order_month').agg(revenue=('revenue','sum'), profit=('profit','sum'), orders=('order_id','nunique')).reset_index()
monthly['revenue_growth_pct'] = monthly['revenue'].pct_change()
monthly.tail(12)


## Category analysis
Compare category and sub-category revenue contribution against profit quality.

In [ ]:

category = df.groupby(['category','sub_category']).agg(
    revenue=('revenue','sum'), profit=('profit','sum'), orders=('order_id','nunique'), margin=('profit_margin','mean')
).sort_values('revenue', ascending=False)
category.head(20)


## Segment analysis
Customer value segmentation reveals where retention and cross-sell investments should focus.

In [ ]:

segments = df.groupby(['segment','customer_value_segment']).agg(
    customers=('customer_id','nunique'), revenue=('revenue','sum'), profit=('profit','sum'), orders=('order_id','nunique')
).reset_index()
segments
